In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.neighbors import kneighbors_graph
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')


# -------------------------- 1. 数据加载与特征处理优化 --------------------------
def load_unsw_nb15(data_path):
    """加载并预处理UNSW-NB15数据集（含特征重要性筛选）"""
    # 加载数据
    df = pd.read_csv(data_path)
    df = df.dropna().drop_duplicates()  # 数据清洗

    # 定义核心特征（确保保留）
    core_numerical = ['sbytes', 'dbytes', 'rate', 'dur', 'spkts', 'dpkts']
    core_categorical = ['proto', 'service', 'state']

    # 分离特征和标签
    all_features = df.drop(['label', 'attack_cat', 'id'], axis=1, errors='ignore').columns.tolist()
    X_all = df[all_features].copy()
    y = df['label'].values

    # 划分数值和分类特征（原始）
    numerical_features = [f for f in all_features if X_all[f].dtype != 'object']
    categorical_features = [f for f in all_features if X_all[f].dtype == 'object']

    # 对分类特征进行临时编码（用于特征重要性评估）
    X_encoded = X_all.copy()
    le_dict = {}
    for f in categorical_features:
        le = LabelEncoder()
        X_encoded[f] = le.fit_transform(X_encoded[f].astype(str))
        le_dict[f] = le  # 保存编码器用于后续处理

    # 特征重要性评估（使用随机森林）
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_encoded, y)
    importances = rf.feature_importances_
    feature_importance = pd.Series(importances, index=all_features).sort_values(ascending=False)

    # 筛选特征：保留核心特征 + 重要性前50%的特征
    top_features = set(core_numerical + core_categorical)  # 确保核心特征保留
    top_features.update(feature_importance.index[:int(len(all_features)*0.5)])  # 加入高重要性特征
    top_numerical = [f for f in numerical_features if f in top_features]
    top_categorical = [f for f in categorical_features if f in top_features]

    # 处理数值特征（标准化）
    X_num = df[top_numerical].values
    scaler = StandardScaler()
    X_num_scaled = scaler.fit_transform(X_num).astype(np.float32)

    # 处理分类特征（保留原始索引，用于嵌入层）
    X_cat_indices = {}  # 存储每个分类特征的索引（未编码为数值）
    cat_sizes = {}  # 每个分类特征的类别数量（用于嵌入层维度）
    for f in top_categorical:
        le = le_dict[f]
        X_cat_indices[f] = le.transform(df[f].astype(str))  # 转为索引（0-based）
        cat_sizes[f] = len(le.classes_)  # 类别数量

    # 检查类别平衡
    normal_count = (y == 0).sum()
    attack_count = (y == 1).sum()
    print(f"数据类别分布：正常样本 {normal_count}，攻击样本 {attack_count}，比例 {normal_count/attack_count:.2f}:1")

    return {
        'X_num': X_num_scaled,  # 标准化数值特征
        'X_cat': X_cat_indices,  # 分类特征索引（用于嵌入）
        'cat_sizes': cat_sizes,  # 分类特征类别数量
        'y': y,
        'attack_cat': df['attack_cat'].values,
        'top_numerical': top_numerical,
        'top_categorical': top_categorical
    }


# -------------------------- 2. 图构建优化 --------------------------
def build_graph_data(processed_data, k_neighbors=5):
    X_num = processed_data['X_num']  # 已为float32
    y = processed_data['y']
    num_nodes = X_num.shape[0]

    # 数值特征张量（强制float32）
    x_num = torch.from_numpy(X_num).float()  # 用from_numpy避免类型转换问题

    # 边权重（强制float32）
    adj = kneighbors_graph(X_num, k_neighbors, mode='distance', include_self=False)
    adj.data = np.exp(-adj.data).astype(np.float32)  # 边权重转为float32
    adj = 0.5 * (adj + adj.T)
    edge_index = []
    edge_attr = []
    for i, j in zip(*adj.nonzero()):
        edge_index.append([i, j])
        edge_attr.append(adj[i, j])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32)  # 明确float32

    # 标签（强制long）
    y_tensor = torch.tensor(y, dtype=torch.long)

    # 划分mask
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    indices = np.random.permutation(num_nodes)
    train_size = int(0.6 * num_nodes)
    val_size = int(0.2 * num_nodes)
    train_mask[indices[:train_size]] = True
    val_mask[indices[train_size:train_size+val_size]] = True
    test_mask[indices[train_size+val_size:]] = True

    # 构建数据对象（所有类型显式指定）
    data = Data(
        x_num=x_num,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y_tensor,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask,
        cat_indices={k: torch.tensor(v, dtype=torch.long) for k, v in processed_data['X_cat'].items()},
        cat_sizes=processed_data['cat_sizes']
    )
    return data


# -------------------------- 3. 模型结构优化（含嵌入层和多头GAT） --------------------------
class MultiModalGNN(torch.nn.Module):
    def __init__(self, num_numerical, cat_info, hidden_dim=128, output_dim=2, num_layers=4, 
                 dropout=0.5, gat_heads=4):
        super(MultiModalGNN, self).__init__()
        self.dropout = dropout

        # 分类特征嵌入层（参数为float32）
        self.cat_embeddings = torch.nn.ModuleDict()
        cat_embed_dim = 16
        total_cat_dim = 0
        for f, size in cat_info.items():
            emb = torch.nn.Embedding(num_embeddings=size, embedding_dim=cat_embed_dim)
            emb.weight.data = emb.weight.data.float()  # 强制权重为float32
            self.cat_embeddings[f] = emb
            total_cat_dim += cat_embed_dim

        # GNN层（参数为float32）
        input_dim = num_numerical + total_cat_dim
        self.gnn_layers = torch.nn.ModuleList()
        self.bn_layers = torch.nn.ModuleList()
        
        # 第一层
        gat = GATConv(input_dim, hidden_dim, heads=gat_heads, concat=True)
        gat.lin.weight.data = gat.lin.weight.data.float()  # 强制权重为float32
        self.gnn_layers.append(gat)
        self.bn_layers.append(torch.nn.BatchNorm1d(hidden_dim * gat_heads))
        current_dim = hidden_dim * gat_heads

        # 后续层
        for _ in range(num_layers - 1):
            gat = GATConv(current_dim, hidden_dim, heads=gat_heads, concat=True)
            gat.lin.weight.data = gat.lin.weight.data.float()
            self.gnn_layers.append(gat)
            self.bn_layers.append(torch.nn.BatchNorm1d(hidden_dim * gat_heads))
            current_dim = hidden_dim * gat_heads

        # 分类器（参数为float32）
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(current_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim, output_dim)
        )
        # 强制分类器参数为float32
        for layer in self.classifier:
            if hasattr(layer, 'weight'):
                layer.weight.data = layer.weight.data.float()

        # 关键节点预测器
        self.key_node_predictor = torch.nn.Sequential(
            torch.nn.Linear(current_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim, 1)
        )
        # 强制预测器参数为float32
        for layer in self.key_node_predictor:
            if hasattr(layer, 'weight'):
                layer.weight.data = layer.weight.data.float()

        self.node_embedding = None


    def forward(self, data):
        # 输入特征强制float32
        x_num = data.x_num.float()
        x_cat_embed = []
        for f in data.cat_indices:
            x_cat_embed.append(self.cat_embeddings[f](data.cat_indices[f]).float())
        x = torch.cat([x_num] + x_cat_embed, dim=1).float()

        # 边属性强制float32
        edge_attr = data.edge_attr.float() if data.edge_attr is not None else None

        # GNN传播
        for i, (gnn, bn) in enumerate(zip(self.gnn_layers, self.bn_layers)):
            x = gnn(x, data.edge_index, edge_attr)
            x = bn(x).float()
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        self.node_embedding = x.float()

        # 输出强制float32
        return {
            'class_pred': self.classifier(x).float(),
            'key_node_scores': self.key_node_predictor(x).float(),
            'node_embedding': x.float()
        }


# -------------------------- 4. 威胁传播预测优化 --------------------------
class ThreatPropagationPredictor:
    def __init__(self, model, data):
        self.model = model
        self.data = data
        self.node_embedding = self._compute_node_embedding()
        self.num_nodes = data.x_num.size(0)

        # 构建邻接矩阵（带权重）
        self.adj_matrix = torch.zeros((self.num_nodes, self.num_nodes), device=data.x_num.device)
        edge_index = data.edge_index.cpu().numpy()
        edge_attr = data.edge_attr.cpu().numpy()
        for i in range(edge_index.shape[1]):
            src, dst = edge_index[0, i], edge_index[1, i]
            self.adj_matrix[src, dst] = torch.tensor(edge_attr[i], device=data.x_num.device)
        self.adj_matrix = self.adj_matrix / (self.adj_matrix.sum(1, keepdim=True) + 1e-10)  # 归一化


    def _compute_node_embedding(self):
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(self.data)
            return outputs['node_embedding'].detach().cpu().numpy()


    def predict_propagation(self, initial_infected_nodes, steps=3, method='attention'):
        """结合图拓扑与节点嵌入的传播预测（要求8、9）"""
        initial_infected_nodes = np.array(initial_infected_nodes).ravel()
        infection_state = np.zeros(self.num_nodes)
        infection_state[initial_infected_nodes] = 1.0
        infection_history = [infection_state.copy()]

        # 边权重矩阵（用于结合拓扑）
        edge_weight = self.adj_matrix.cpu().numpy()

        for step in range(steps):
            if method == 'attention':
                # 计算节点相似度
                infected_embeddings = self.node_embedding[initial_infected_nodes]
                similarity = np.zeros((len(initial_infected_nodes), self.num_nodes))
                for i, idx in enumerate(initial_infected_nodes):
                    embed = self.node_embedding[idx].reshape(-1)
                    dot_product = np.dot(self.node_embedding, embed)
                    norm_product = np.linalg.norm(self.node_embedding, axis=1) * np.linalg.norm(embed)
                    similarity[i] = dot_product / np.maximum(norm_product, 1e-10)
                similarity = np.mean(similarity, axis=0)

                # 传播概率 = 相似度 * 边权重（要求8）
                new_infection_state = (similarity * edge_weight[initial_infected_nodes].mean(axis=0)) 
                new_infection_state = np.clip(new_infection_state, 0, 1)  # 限制在[0,1]

            # 更新感染状态
            infection_state = new_infection_state
            infection_history.append(infection_state.copy())

            # 限制传播范围（绝对阈值>0.6，要求9）
            if step < steps - 1:
                new_infected = np.where(infection_state > 0.6)[0]  # 绝对阈值
                initial_infected_nodes = np.unique(np.concatenate([initial_infected_nodes, new_infected]))

        return np.array(infection_history)


# -------------------------- 5. 训练与评估优化 --------------------------
def train(model, data, optimizer, criterion, device, propagation_loss_weight=0.3):
    model.train()
    optimizer.zero_grad()
    
    # 模型输出强制float32并移至设备
    outputs = model(data)
    class_pred = outputs['class_pred'].to(device).float()
    key_node_scores = outputs['key_node_scores'].to(device).float()

    # 标签强制long并移至设备
    y_train = data.y[data.train_mask].to(device).long()
    class_pred_train = class_pred[data.train_mask].float()  # 再次确认

    # 计算损失（此时必为Float预测 + Long标签）
    class_loss = criterion(class_pred_train, y_train)

    # 关键节点损失（同上）
    attack_indices = torch.where(data.y[data.train_mask] == 1)[0]
    if len(attack_indices) > 0:
        top_k = max(1, int(len(attack_indices) * 0.1))
        train_attack_idx = torch.where(data.train_mask)[0][attack_indices]
        key_indices = np.random.choice(train_attack_idx.cpu().numpy(), top_k, replace=False)
        key_mask = torch.zeros_like(data.y, dtype=torch.bool)
        key_mask[key_indices] = True

        key_loss = F.binary_cross_entropy_with_logits(
            key_node_scores[key_mask].squeeze().float(),
            torch.ones_like(key_node_scores[key_mask].squeeze(), device=device).float()
        )
        non_key_loss = F.binary_cross_entropy_with_logits(
            key_node_scores[~key_mask].squeeze().float(),
            torch.zeros_like(key_node_scores[~key_mask].squeeze(), device=device).float()
        )
        propagation_loss = key_loss + non_key_loss
    else:
        propagation_loss = torch.tensor(0.0, device=device).float()

    total_loss = class_loss + propagation_loss_weight * propagation_loss
    total_loss.backward()
    optimizer.step()
    return total_loss.item()


def evaluate(model, data, device):
    model.eval()
    with torch.no_grad():
        outputs = model(data)
        class_pred = outputs['class_pred'].argmax(dim=1)
        y_true = data.y.cpu().numpy()
        y_pred = class_pred.cpu().numpy()

        # 计算准确率
        train_acc = (y_pred[data.train_mask.cpu()] == y_true[data.train_mask.cpu()]).mean()
        val_acc = (y_pred[data.val_mask.cpu()] == y_true[data.val_mask.cpu()]).mean()
        test_acc = (y_pred[data.test_mask.cpu()] == y_true[data.test_mask.cpu()]).mean()
        rpa = f1_score(y_true[data.test_mask.cpu()], y_pred[data.test_mask.cpu()])

    return {'train_acc': train_acc, 'val_acc': val_acc, 'test_acc': test_acc, 'rpa': rpa}


# -------------------------- 6. 主函数 --------------------------
def main():
    # 加载数据
    data_path = r'D:\code\github\kaggle_data\mrwellsdavid\unsw-nb15\versions\1\UNSW_NB15_training-set.csv'
    processed_data = load_unsw_nb15(data_path)
    data = build_graph_data(processed_data, k_neighbors=5)

    # 设备配置（确保设备上的张量为float32）
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data = data.to(device)
    for f in data.cat_indices:
        data.cat_indices[f] = data.cat_indices[f].to(device).long()  # 分类索引强制long

    # 损失函数（权重强制float32）
    y = processed_data['y']
    normal_ratio = (y == 0).mean()
    weight = torch.tensor([normal_ratio, 1 - normal_ratio], device=device).float()  # 权重强制float32
    criterion = torch.nn.CrossEntropyLoss(weight=weight)

    # 初始化模型
    model = MultiModalGNN(
        num_numerical=processed_data['X_num'].shape[1],
        cat_info=processed_data['cat_sizes'],
        hidden_dim=128,
        num_layers=4,
        dropout=0.5,
        gat_heads=4
    ).to(device)
    # 强制模型参数为float32
    model = model.float()

    # 优化器
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

    # 训练循环
    best_val_acc = 0
    patience = 15
    counter = 0
    for epoch in range(1, 201):
        loss = train(model, data, optimizer, criterion, device)
        metrics = evaluate(model, data, device)
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train: {metrics["train_acc"]:.4f}, Val: {metrics["val_acc"]:.4f}')

        if metrics['val_acc'] > best_val_acc:
            best_val_acc = metrics['val_acc']
            torch.save(model.state_dict(), 'best_model.pt')
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print('Early stopping!')
                break


def evaluate(model, data, device):
    model.eval()
    with torch.no_grad():
        outputs = model(data)
        class_pred = outputs['class_pred'].argmax(dim=1).cpu().numpy()
        y_true = data.y.cpu().numpy()
        train_acc = (class_pred[data.train_mask.cpu()] == y_true[data.train_mask.cpu()]).mean()
        val_acc = (class_pred[data.val_mask.cpu()] == y_true[data.val_mask.cpu()]).mean()
        test_acc = (class_pred[data.test_mask.cpu()] == y_true[data.test_mask.cpu()]).mean()
        rpa = f1_score(y_true[data.test_mask.cpu()], class_pred[data.test_mask.cpu()])
    return {'train_acc': train_acc, 'val_acc': val_acc, 'test_acc': test_acc, 'rpa': rpa}


if __name__ == '__main__':
    main()

In [ ]:
model.load_state_dict(torch.load('best_model.pt'))
test_metrics = evaluate(model, data, device)
print(f"Test Acc: {test_metrics['test_acc']:.4f}, RPA: {test_metrics['rpa']:.4f}")